In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import pandas as pd
import plotly.express as px
import yaml
from dotenv import load_dotenv

from my_project.data.utils import columns_info_func, load_column_config

pd.options.display.max_columns = None

In [ ]:
load_dotenv(override=True)

CONFIGS_DIR = os.getenv("CONFIGS_DIR")
DATA_DIR = os.getenv("DATA_DIR")
RESULTS_DIR = os.getenv("RESULTS_DIR")

In [ ]:
with open(os.path.join(CONFIGS_DIR, "data.yaml")) as f:
    data_config = yaml.safe_load(f)

col_cfg = load_column_config(CONFIGS_DIR)

In [ ]:
df = pd.read_csv(
    os.path.join(DATA_DIR, data_config["paths"]["raw_data_path"]), sep=";", decimal=","
)
df = df.set_index(col_cfg.ID)

In [ ]:
df_transformed = df.copy()
print(f"Dane wejściowe: {df_transformed.shape}")

In [ ]:
MAX_PROPORTION_THRESHOLD = 0.95

# Data Processing

## Rename

In [ ]:
df = df.rename(
    columns={"'ZgonJakikolwiek(nawet>12miesiecy)": "ZgonJakikolwiek>12miesiecy"}
)

## Medical categories

In [ ]:
# LBBB/RBBB category based on numeric code
df_transformed["LBBB_RBBB"] = df_transformed["LBBB=1,RBBB=2,norma=0,inne=3"].map(
    {0: "norma", 1: "LBBB", 2: "RBBB", 3: "inne"}
)

# Place of death category
df_transformed["Zgon"] = (
    df_transformed["Zgon1=szp,2=poza,0=zyje"]
    .map({0: "zyje", 1: "szpital", 2: "poza szpitalem"})
    .fillna("żyje")
)

# Time of death category
df_transformed["Zgon_kiedy"] = None
df_transformed.loc[df_transformed["Zgon_do_6_miesiecy"] == 1, "Zgon_kiedy"] = (
    "do_6_miesiecy"
)
df_transformed.loc[df_transformed["Zgon_6-12_miesiecy"] == 1, "Zgon_kiedy"] = (
    "6-12_miesiecy"
)

mask = (df_transformed["ZgonJakikolwiek(nawet>12miesiecy)"] == 1) & (
    df_transformed["Zgon_kiedy"].isna()
)
df_transformed.loc[mask, "Zgon_kiedy"] = "Jakikolwiek(nawet>12miesiecy)"

df_transformed["Zgon_kiedy"] = df_transformed["Zgon_kiedy"].fillna("Zyje").astype("str")

# Diagnosis category reduction

In [ ]:
var = "Rozpoznanie_Glowne"

In [ ]:
df_transformed[var + "_main"] = df_transformed[var].str.slice(0, 3)

In [ ]:
df_aggr = (
    df_transformed.groupby([var + "_main", var])
    .size()
    .rename("count")
    .to_frame()
    .sort_index()
)

df_aggr_group = df_aggr.reset_index().groupby(var + "_main")["count"].sum().to_frame()

df_aggr["pct"] = df_aggr["count"] / df_aggr["count"].sum()
df_aggr["pct_within_group"] = (
    (df_aggr["count"] / df_aggr_group["count"]).to_frame().sort_index()
)

In [ ]:
selected = df_aggr[
    (df_aggr["pct_within_group"] > 0.1)
    & (df_aggr["pct_within_group"] < 0.9)
    & (df_aggr["count"] > 1)
]
selected

In [ ]:
preserve_as_original = selected.index.get_level_values(var).tolist()

In [ ]:
df_transformed[var] = df_transformed.apply(
    lambda row: row[var] if row[var] in preserve_as_original else row[var + "_main"],
    axis=1,
)

df_transformed = df_transformed.drop(columns=[var + "_main"])

## Date processing

In [ ]:
# Cleaning the data_zg column - special text values
df_transformed.loc[df_transformed["data_zg"] == "ZGON SZPITAL", "data_zg"] = (
    df_transformed.loc[df_transformed["data_zg"] == "ZGON SZPITAL", "Data_Wypisu"]
)
df_transformed.loc[df_transformed["data_zg"] == "Zyje", "data_zg"] = pd.NA

# Converting Excel serial dates to datetime.date
date_columns = [
    "DataPrzyjecia",
    "DataWypisu",
    "KoronarografiaData",
    "Data_Przyjecia",
    "Data_Wypisu",
    "data_zg",
]

for col in date_columns:
    if col in df_transformed.columns:
        # Cast to object to handle mixed types
        df_transformed[col] = df_transformed[col].astype("object")

        # Find numeric values (Excel serial dates)
        numeric_mask = pd.to_numeric(df_transformed[col], errors="coerce").notna()

        if numeric_mask.any():
            # Convert numeric values to dates
            numeric_values = pd.to_numeric(df_transformed[col], errors="coerce")
            dates = pd.to_datetime(
                numeric_values, unit="D", origin="1899-12-30", errors="coerce"
            ).dt.date
            df_transformed.loc[numeric_mask, col] = dates[numeric_mask]

        # Convert remaining values to datetime.date
        df_transformed[col] = pd.to_datetime(
            df_transformed[col], errors="coerce"
        ).dt.date

## Drop columns

In [ ]:
df_transformed = df_transformed.drop(columns=date_columns)

# Target variable

In [ ]:
df_transformed["zgon_binary"] = df_transformed["Zgon_kiedy"] == "do_6_miesiecy"

In [ ]:
df_transformed.drop(columns=["Zgon_kiedy", "Zgon"], inplace=True)

## Value standardization

## Data type corrections

In [ ]:
df_dtypes = df_transformed.dtypes
bool_columns = df_dtypes[df_dtypes == "bool"].index.tolist()
numeric_columns = df_dtypes[
    (df_dtypes == "float64") | (df_dtypes == "int64")
].index.tolist()

In [ ]:
unique_vals_df = (
    df_transformed.apply(lambda x: list(set(x.unique())), axis=0)
    .rename("value")
    .to_frame()
)
unique_vals_df["len"] = unique_vals_df["value"].apply(len)
unique_vals_df["length_without_na"] = unique_vals_df["value"].apply(
    lambda x: len([v for v in x if pd.notna(v)])
)

### Boolean columns

In [ ]:
df_transformed[bool_columns] = df_transformed[bool_columns].astype("boolean")

### Binary columns

In [ ]:
binary_columns = unique_vals_df[
    (unique_vals_df["len"] <= 3) & (unique_vals_df["length_without_na"] <= 2)
].index.tolist()
columns = list(set(binary_columns).intersection(set(numeric_columns)))
df_transformed[columns] = df_transformed[columns].astype("boolean")

### Fix for "rehabilitacja" column

In [ ]:
df_transformed["rehabilitacja"].value_counts(dropna=False)

In [ ]:
df_transformed["rehabilitacja"] = df_transformed["rehabilitacja"].fillna(False)

### NaN normalization

In [ ]:
df_transformed = df_transformed.replace({pd.NA: pd.NA, pd.NaT: pd.NA})

# Drop low cardinality columns

In [ ]:
columns_info = columns_info_func(df_transformed).sort_values(
    "max_proportion", ascending=False
)

In [ ]:
# columns_info.head(10)

In [ ]:
df_plot = columns_info[["max_proportion"]].reset_index()
df_plot["n"] = [1] * len(df_plot)
df_plot["cumulative"] = df_plot["n"].cumsum() / len(df_plot)
px.line(df_plot, x="cumulative", y="max_proportion")

In [ ]:
df_transformed = df_transformed.drop(
    columns=columns_info[
        columns_info["max_proportion"] > MAX_PROPORTION_THRESHOLD
    ].index
)

# Overwiew of the data after cleaning

In [ ]:
df_dtypes = df_transformed.dtypes
df_dtypes.value_counts()

In [ ]:
df_transformed.shape

In [ ]:
df_transformed["powiazano"].value_counts(dropna=False)

In [ ]:
df_transformed.columns.to_list()

# Saving processed data

In [ ]:
# Final column selection. Edit configs/columns.py (comment lines) to control
# which columns are kept. Intersection preserves df order and tolerates columns
# already removed by earlier steps.
col_cfg = load_column_config(CONFIGS_DIR)
keep = [c for c in df_transformed.columns if c in set(col_cfg.KEEP_AFTER_CLEANING)]

dropped = [c for c in df_transformed.columns if c not in keep]
print(f"Dropped {len(dropped)} columns not in whitelist: {dropped}")

df_transformed = df_transformed[keep]
df_transformed.shape

In [ ]:
df_transformed.to_pickle(
    os.path.join(DATA_DIR, data_config["paths"]["preprocessed_data_path"])
)